# 01. Persiapan dan Pembagian Data

Membaca data survei CDC NHANES, verifikasi kelas target, deduplikasi, dan pembagian train-test 80:20.

**Prasyarat:** Dataset asli tersedia. Tidak memerlukan notebook sebelumnya.

**Keluaran:** `cleaned.csv`, `train.csv`, `test.csv`, `notebook_split.json`, dan `cv_folds.csv`.

Pilih kernel Python 3.12 dengan dependency proyek, lalu **Restart Kernel → Run All**. Tahap 01–05 dijalankan secara berurutan. Proyek edukasi data science, bukan diagnosis medis klinis.


## A. Inisialisasi Lingkungan dan Lokasi Proyek
Mempersiapkan konstanta fitur dan folder kerja.


In [1]:
from pathlib import Path
import hashlib
import importlib.metadata
import json
import platform
from datetime import datetime, timezone
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Image, Markdown

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'Dataset/Data_Asli/nhanes_depression_raw.csv').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari dalam folder proyek lengkap dengan Dataset.')
TARGET = "Depression"
LABELS = ["Not Depressed", "Depressed"]
DISPLAY = {
    "Not Depressed": "Tidak Terindikasi Depresi Mayor",
    "Depressed": "Berisiko Mengalami Depresi (PHQ-9 ≥ 10)"
}

CATEGORICAL = [
    "Gender", "Education Level", "Marital Status", "Household Income",
    "Trouble Sleeping History", "Vigorous Recreation", "Moderate Recreation",
    "Current Smoker", "Memory Problems", "Cant Work", "Limited Work"
]
NUMERIC = [
    "Age", "Sleep Hours", "Sedentary Time", "BMI", "Systolic", "Diastolic", "Pulse"
]
FEATURES = CATEGORICAL + NUMERIC

RANGES = {
    "Age": (18, 120),
    "Sleep Hours": (1.0, 24.0),
    "Sedentary Time": (0.0, 1440.0),
    "BMI": (10.0, 90.0),
    "Systolic": (60.0, 260.0),
    "Diastolic": (30.0, 160.0),
    "Pulse": (30.0, 220.0),
}

SEED = 42
REPORTS = ROOT / "Source_Code/reports"
EDA = REPORTS / "eda"
MODELING = REPORTS / "modeling"
ANALYSIS = REPORTS / "analysis"
PROCESSED = ROOT / "Dataset/Data_Olahan"
MODELS = ROOT / "Model"


for folder in [EDA, MODELING, ANALYSIS, PROCESSED, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)
source = ROOT / 'Dataset/Data_Asli/nhanes_depression_raw.csv'
def save_json(path, value):
    path.write_text(
        json.dumps(value, indent=2, ensure_ascii=False,
                   default=lambda x: x.item() if isinstance(x, np.generic) else str(x)),
        encoding="utf-8"
    )

def save_figure(path):
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches='tight')
    display(Image(filename=str(path)))
    plt.close()

print('Proyek:', ROOT)


Proyek: D:\KULIAH\sem-5\MACHINE LEARNING\PJBL\prediksi_gangguan_depresi


## B. Membaca Dataset Asli
Memuat 36.259 baris data survei kesehatan dan memeriksa sebaran kelas awal.


In [2]:
for folder in [EDA, MODELING, ANALYSIS, PROCESSED, MODELS]:
    folder.mkdir(parents=True, exist_ok=True)

source = ROOT / "Dataset/Data_Asli/nhanes_depression_raw.csv"
raw = pd.read_csv(source, low_memory=False)

if set(raw[TARGET].unique()) != set(LABELS):
    raise ValueError("Label dataset tidak sesuai dengan kontrak kelas target.")
display(raw.head(), raw[TARGET].value_counts())


,SEQN,Depression,Age,Gender,Education Level,Marital Status,Household Income,Sleep Hours,Trouble Sleeping History,Sedentary Time,Vigorous Recreation,Moderate Recreation,Current Smoker,Memory Problems,Cant Work,Limited Work,BMI,Systolic,Diastolic,Pulse
0,31131,Not Depressed,44,Female,Some College or AA Degree,Married,$75K+,9.0,No,150.0,No,No,Missing,No,No,No,30.90,144.0,74.0,58.0
1,31132,Not Depressed,70,Male,College Graduate or Above,Married,$75K+,7.0,No,150.0,No,Yes,Missing,No,No,No,24.74,138.0,60.0,62.0
2,31134,Not Depressed,73,Male,High School,Married,Below $25K,7.0,Yes,90.0,No,Yes,Missing,No,No,No,30.63,130.0,68.0,50.0
3,31139,Not Depressed,18,Female,Missing,Never Married,$75K+,6.0,No,120.0,Yes,Yes,Missing,No,No,No,29.45,110.0,64.0,68.0
4,31143,Not Depressed,19,Male,Missing,Never Married,$75K+,7.0,No,360.0,Yes,No,Missing,No,No,No,22.57,108.0,62.0,68.0


Depression
Not Depressed    33091
Depressed         3168
Name: count, dtype: int64

## C. Pembersihan Fitur dan Integritas Data
Mengekstrak 18 fitur klinis dan memeriksa ada/tidaknya duplikat.


In [3]:
def prepare_features(frame):
    """Transformasi deterministik input tanpa membocorkan statistik data uji/global."""
    data = frame.copy()
    missing = sorted(set(FEATURES) - set(data.columns))
    if missing:
        raise ValueError("Kolom wajib belum tersedia: " + ", ".join(missing))
    data = data[FEATURES].copy()
    for col in CATEGORICAL:
        data[col] = data[col].astype(str).replace(r"^\s*$", np.nan, regex=True)
    for col in NUMERIC:
        original = data[col]
        data[col] = pd.to_numeric(original, errors="coerce")
        invalid = original.notna() & original.astype(str).str.strip().ne("") & data[col].isna()
        if invalid.any():
            raise ValueError(f"Kolom {col} berisi nilai yang tidak dapat diubah menjadi numerik.")
        if np.isinf(data[col].to_numpy(dtype=float)).any():
            raise ValueError(f"Kolom {col} tidak boleh berisi nilai infinity.")
    return data


features_all = prepare_features(raw)
canonical = features_all.assign(**{TARGET: raw[TARGET]})
duplicate_mask = canonical.duplicated(keep="first")
raw.loc[duplicate_mask].to_csv(EDA / "duplicate_records.csv", index=False)

clean = canonical.loc[~duplicate_mask].copy().reset_index(drop=True)
ids = raw.loc[~duplicate_mask, "SEQN"].reset_index(drop=True)

# Deteksi profil identik dengan label bertentangan jika ada
profile_hashes = pd.util.hash_pandas_object(clean[FEATURES], index=False).astype(str)
conflict_counts = clean.assign(profile_hash=profile_hashes).groupby("profile_hash")[TARGET].nunique()
conflict_hashes = set(conflict_counts[conflict_counts > 1].index)
clean.assign(profile_hash=profile_hashes).loc[profile_hashes.isin(conflict_hashes)].to_csv(
    EDA / "conflicting_profiles.csv", index=False
)

clean.assign(source_seqn=ids, profile_hash=profile_hashes).to_csv(
    PROCESSED / "cleaned.csv", index=False
)

X = clean[FEATURES]
y = clean[TARGET]
print('Baris bersih:', len(clean), '| Duplikat:', int(duplicate_mask.sum()))
display(clean.head())


Baris bersih: 36259 | Duplikat: 0


,Gender,Education Level,Marital Status,Household Income,Trouble Sleeping History,Vigorous Recreation,Moderate Recreation,Current Smoker,Memory Problems,Cant Work,Limited Work,Age,Sleep Hours,Sedentary Time,BMI,Systolic,Diastolic,Pulse,Depression
0,Female,Some College or AA Degree,Married,$75K+,No,No,No,Missing,No,No,No,44,9.0,150.0,30.90,144.0,74.0,58.0,Not Depressed
1,Male,College Graduate or Above,Married,$75K+,No,No,Yes,Missing,No,No,No,70,7.0,150.0,24.74,138.0,60.0,62.0,Not Depressed
2,Male,High School,Married,Below $25K,Yes,No,Yes,Missing,No,No,No,73,7.0,90.0,30.63,130.0,68.0,50.0,Not Depressed
3,Female,Missing,Never Married,$75K+,No,Yes,Yes,Missing,No,No,No,18,6.0,120.0,29.45,110.0,64.0,68.0,Not Depressed
4,Male,Missing,Never Married,$75K+,No,Yes,No,Missing,No,No,No,19,7.0,360.0,22.57,108.0,62.0,68.0,Not Depressed


## D. Pembagian Data Latih dan Uji (80:20 Stratified)
Mengunci pembagian berstrata dengan seed 42 dan menyiapkan 5-fold CV.


In [4]:
from sklearn.model_selection import StratifiedKFold, train_test_split

# Split train-test 80:20 berstrata
X_train, X_test, y_train, y_test, train_idx, test_idx = train_test_split(
    X, y, clean.index, test_size=0.20, stratify=y, random_state=SEED
)

# K-Fold Cross Validation berstrata 5 fold pada data latih
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED + 1)
fold_records = []
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train), 1):
    for role, indexes in [("fit", fit_idx), ("validation", val_idx)]:
        for pos in indexes:
            fold_records.append({
                "fold": fold, "role": role,
                "clean_index": int(X_train.index[pos]),
                "label": y_train.iloc[pos]
            })
pd.DataFrame(fold_records).to_csv(MODELING / "cv_folds.csv", index=False)

for name, indices in [("train", train_idx), ("test", test_idx)]:
    clean.loc[indices].assign(source_seqn=ids.loc[indices]).to_csv(
        PROCESSED / f"{name}.csv", index=False
    )
print('Train:', len(train_idx), '| Test:', len(test_idx))


Train: 29007 | Test: 7252


## E. Menyimpan State Handoff Antar-Notebook
Menyimpan indeks dan hash integritas data.


In [5]:
input_paths = [source, PROCESSED / 'cleaned.csv', PROCESSED / 'train.csv', PROCESSED / 'test.csv']
split_state = {
    'train_indices': train_idx.tolist(), 'test_indices': test_idx.tolist(),
    'input_hashes': {p.relative_to(ROOT).as_posix(): hashlib.sha256(p.read_bytes()).hexdigest() for p in input_paths},
    'seed': SEED,
}
save_json(PROCESSED / 'notebook_split.json', split_state)
print('Tahap 01 Selesai. Lanjut ke 02_eksplorasi_data.ipynb')


Tahap 01 Selesai. Lanjut ke 02_eksplorasi_data.ipynb
